In [2]:
import pandas as pd
print("It works!", pd.__version__)

It works! 3.0.6


In [3]:
df = pd.read_csv("players_data-2025_2026.csv")
df.head()

,Rk,Player,Nation,Pos,Squad,Comp,Age,Born,MP,Starts,...,CrdY_stats_misc,CrdR_stats_misc,2CrdY,Fls,Fld,Off,Crs,Int,TklW,OG
0,1,Brenden Aaronson,us USA,"MF,FW",Leeds United,eng Premier League,24.0,2000.0,37,30,...,3,0,0,20,51,5,37,17,27,0
1,2,Jerome Abbey,eng ENG,MF,Wolves,eng Premier League,15.0,2009.0,1,0,...,0,0,0,0,0,1,0,1,0,0
2,3,Zach Abbott,eng ENG,DF,Nottingham Forest,eng Premier League,19.0,2006.0,3,2,...,0,0,0,3,1,0,3,2,4,0
3,4,Jones El-Abdellaoui,ma MAR,"MF,FW",Celta Vigo,es La Liga,19.0,2006.0,22,4,...,0,0,0,7,3,2,30,3,3,0
4,5,Himad Abdelli,dz ALG,MF,Marseille,fr Ligue 1,25.0,1999.0,8,1,...,1,0,0,3,0,0,1,2,1,0


In [4]:
print(df.shape)
print(list(df.columns))

(2839, 102)
['Rk', 'Player', 'Nation', 'Pos', 'Squad', 'Comp', 'Age', 'Born', 'MP', 'Starts', 'Min', '90s', 'Gls', 'Ast', 'G+A', 'G-PK', 'PK', 'PKatt', 'CrdY', 'CrdR', 'G+A-PK', 'Rk_stats_keeper', 'Nation_stats_keeper', 'Pos_stats_keeper', 'Comp_stats_keeper', 'Age_stats_keeper', 'Born_stats_keeper', 'MP_stats_keeper', 'Starts_stats_keeper', 'Min_stats_keeper', '90s_stats_keeper', 'GA', 'GA90', 'SoTA', 'Saves', 'Save%', 'W', 'D', 'L', 'CS', 'CS%', 'PKatt_stats_keeper', 'PKA', 'PKsv', 'PKm', 'Rk_stats_shooting', 'Nation_stats_shooting', 'Pos_stats_shooting', 'Comp_stats_shooting', 'Age_stats_shooting', 'Born_stats_shooting', '90s_stats_shooting', 'Gls_stats_shooting', 'Sh', 'SoT', 'SoT%', 'Sh/90', 'SoT/90', 'G/Sh', 'G/SoT', 'PK_stats_shooting', 'PKatt_stats_shooting', 'Rk_stats_playing_time', 'Nation_stats_playing_time', 'Pos_stats_playing_time', 'Comp_stats_playing_time', 'Age_stats_playing_time', 'Born_stats_playing_time', 'MP_stats_playing_time', 'Min_stats_playing_time', 'Mn/MP', 'M

In [6]:
print(df["Comp"].value_counts())
print(df["Pos"].value_counts().head(10))
print(df["Min"].describe())

Comp
es La Liga            611
it Serie A            609
fr Ligue 1            561
eng Premier League    551
de Bundesliga         507
Name: count, dtype: int64
Pos
MF       922
DF       705
FW       398
MF,FW    217
GK       194
FW,MF    144
DF,MF    141
MF,DF    115
DF,FW      3
Name: count, dtype: int64
count    2839.000000
mean     1218.580134
std       963.218994
min         1.000000
25%       320.500000
50%      1074.000000
75%      1972.500000
max      3420.000000
Name: Min, dtype: float64


In [7]:
df["PosGroup"] = df["Pos"].str.split(",").str[0]
print(df["PosGroup"].value_counts())

for m in [450, 900, 1350]:
    print(m, "minutes:", (df["Min"] >= m).sum(), "players left")

PosGroup
MF    1254
DF     849
FW     542
GK     194
Name: count, dtype: int64
450 minutes: 1999 players left
900 minutes: 1578 players left
1350 minutes: 1192 players left


In [8]:
friendly = {
    "Gls": "Goals",
    "Ast": "Assists",
    "Int": "Interceptions",
    "TklW": "TacklesWon",
    "Fls": "FoulsCommitted",
    "Fld": "FoulsDrawn",
    "Crs": "Crosses",
    "Off": "Offsides",
}

players = df[df["Min"] >= 900].copy()

for short, name in friendly.items():
    players[name + "_per90"] = players[short] / players["90s"]

players[["Player", "Squad", "PosGroup", "Goals_per90", "Assists_per90", "TacklesWon_per90"]].head(10)

,Player,Squad,PosGroup,Goals_per90,Assists_per90,TacklesWon_per90
0,Brenden Aaronson,Leeds United,MF,0.147059,0.183824,0.992647
5,Himad Abdelli,Angers,MF,0.190476,0.000000,1.619048
6,Ali Abdi,Nice,MF,0.227273,0.000000,0.833333
7,Salis Abdul Samed,Nice,MF,0.000000,0.000000,0.693069
8,Saud Abdulhamid,Lens,MF,0.133333,0.266667,1.466667
10,Laurent Abergel,Lorient,MF,0.038760,0.000000,0.891473
11,Matthis Abline,Nantes,FW,0.209059,0.139373,0.452962
12,Abner,Lyon,DF,0.125523,0.083682,1.004184
14,Abdel Abqar,Getafe,DF,0.000000,0.125000,1.062500
19,Francesco Acerbi,Inter,DF,0.000000,0.065359,0.588235


In [9]:
per90_cols = [name + "_per90" for name in friendly.values()]

for col in per90_cols:
    players[col + "_percentile"] = players.groupby("PosGroup")[col].rank(pct=True) * 100

forwards = players[players["PosGroup"] == "FW"]
forwards[["Player", "Squad", "Goals_per90", "Goals_per90_percentile"]].sort_values("Goals_per90", ascending=False).head(10)

,Player,Squad,Goals_per90,Goals_per90_percentile
1319,Harry Kane,Bayern Munich,1.363636,100.000000
406,Jonathan Burkardt,Eintracht Frankfurt,0.878378,99.567100
1701,Kylian Mbappé,Real Madrid,0.865052,99.134199
1610,Donyell Malen,Roma,0.858896,98.701299
667,Ousmane Dembélé,Paris Saint-Germain,0.847458,98.268398
1104,Erling Haaland,Manchester City,0.823171,97.835498
2656,Deniz Undav,Stuttgart,0.763052,97.402597
1491,Robert Lewandowski,Barcelona,0.756757,96.969697
713,Bamba Dieng,Lorient,0.740741,96.536797
833,Carlos Espí,Levante,0.738255,96.103896


In [10]:
lower_is_better = ["FoulsCommitted", "Offsides"]

for name in lower_is_better:
    col = name + "_per90_percentile"
    players[col] = 100 - players[col]

defenders = players[players["PosGroup"] == "DF"]
defenders[["Player", "Squad", "FoulsCommitted_per90", "FoulsCommitted_per90_percentile"]].sort_values("FoulsCommitted_per90", ascending=False).head(5)

,Player,Squad,FoulsCommitted_per90,FoulsCommitted_per90_percentile
2546,Nicolás Tagliafico,Lyon,2.465753,0.000000
137,Jon Aramburu,Real Sociedad,2.240260,0.189036
2137,Marin Pongračić,Fiorentina,2.129630,0.378072
1629,Marcão,Sevilla,2.115385,0.567108
1256,Juan Jesus,Napoli,2.096774,0.756144


In [11]:
team = df.groupby(["Squad", "Comp"]).agg(
    Goals=("Gls", "sum"),
    Assists=("Ast", "sum"),
    Shots=("Sh", "sum"),
    ShotsOnTarget=("SoT", "sum"),
    Interceptions=("Int", "sum"),
    TacklesWon=("TklW", "sum"),
    Crosses=("Crs", "sum"),
    FoulsCommitted=("Fls", "sum"),
    Offsides=("Off", "sum"),
    GoalsConceded=("GA", "sum"),
    TotalPlayer90s=("90s", "sum")
).reset_index()

team["Matches"] = (team["TotalPlayer90s"] / 11).round()

print(team.shape)
print(team["Matches"].value_counts())

(96, 14)
Matches
38.0    60
34.0    36
Name: count, dtype: int64


In [13]:
team_stats = ["Goals", "Assists", "Shots", "ShotsOnTarget", "Interceptions",
              "TacklesWon", "Crosses", "FoulsCommitted", "Offsides", "GoalsConceded"]
team_lower_is_better = ["FoulsCommitted", "Offsides", "GoalsConceded"]

for s in team_stats:
    team[s + "_permatch"] = team[s] / team["Matches"]
    team[s + "_permatch_percentile"] = team[s + "_permatch"].rank(pct=True) * 100

for s in team_lower_is_better:
    col = s + "_permatch_percentile"
    team[col] = 100 - team[col]

print("Best defences (fewest goals conceded per match):")
team.sort_values("GoalsConceded_permatch")[["Squad", "Comp", "GoalsConceded_permatch", "GoalsConceded_permatch_percentile"]].head(5)

Best defences (fewest goals conceded per match):


,Squad,Comp,GoalsConceded_permatch,GoalsConceded_permatch_percentile
2,Arsenal,eng Premier League,0.710526,98.958333
20,Como,it Serie A,0.763158,97.916667
78,Roma,it Serie A,0.815789,96.875000
69,Paris Saint-Germain,fr Ligue 1,0.852941,95.833333
40,Juventus,it Serie A,0.894737,94.791667


In [15]:
def team_profile(name):
    row = team[team["Squad"] == name].iloc[0]
    print(f"{row['Squad']} ({row['Comp']})\n")

    results = []
    for s in team_stats:
        results.append((s, row[s + "_permatch"], row[s + "_permatch_percentile"]))
    results.sort(key=lambda x: x[2], reverse=True)

    for s, value, pct in results:
        if pct >= 75:
            tag = "STRENGTH"
        elif pct <= 25:
            tag = "WEAKNESS"
        else:
            tag = "average"
        print(f"{s:16} {value:6.2f} per match | Percentile: {pct:5.1f} | {tag}")

team_profile("Arsenal")

Arsenal (eng Premier League)

GoalsConceded      0.71 per match | Percentile:  99.0 | STRENGTH
Assists            1.34 per match | Percentile:  87.5 | STRENGTH
Goals              1.76 per match | Percentile:  84.4 | STRENGTH
Shots             14.53 per match | Percentile:  84.4 | STRENGTH
ShotsOnTarget      4.89 per match | Percentile:  82.3 | STRENGTH
FoulsCommitted    10.29 per match | Percentile:  78.1 | STRENGTH
Crosses           17.87 per match | Percentile:  58.9 | average
Offsides           1.61 per match | Percentile:  57.3 | average
TacklesWon         8.74 per match | Percentile:  19.8 | WEAKNESS
Interceptions      6.82 per match | Percentile:   9.4 | WEAKNESS


In [18]:
team[team["Squad"].str.contains("Man", case=False)]["Squad"].tolist()

['Manchester City', 'Manchester Utd']

In [19]:
team_profile("Manchester Utd")

Manchester Utd (eng Premier League)

Shots             15.66 per match | Percentile:  91.1 | STRENGTH
ShotsOnTarget      5.66 per match | Percentile:  90.6 | STRENGTH
TacklesWon        10.68 per match | Percentile:  83.9 | STRENGTH
Goals              1.74 per match | Percentile:  83.3 | STRENGTH
FoulsCommitted    10.26 per match | Percentile:  79.7 | STRENGTH
Assists            1.21 per match | Percentile:  79.2 | STRENGTH
Offsides           1.37 per match | Percentile:  75.5 | STRENGTH
GoalsConceded      1.32 per match | Percentile:  64.1 | average
Interceptions      8.16 per match | Percentile:  48.4 | average
Crosses           16.82 per match | Percentile:  37.0 | average


In [20]:
team_profile("Chelsea")

Chelsea (eng Premier League)

Interceptions      9.42 per match | Percentile:  89.6 | STRENGTH
Shots             13.39 per match | Percentile:  74.0 | average
Goals              1.50 per match | Percentile:  69.8 | average
Assists            1.03 per match | Percentile:  69.8 | average
Offsides           1.53 per match | Percentile:  64.1 | average
FoulsCommitted    11.03 per match | Percentile:  60.4 | average
ShotsOnTarget      4.37 per match | Percentile:  57.8 | average
Crosses           17.82 per match | Percentile:  56.8 | average
GoalsConceded      1.42 per match | Percentile:  45.3 | average
TacklesWon         8.95 per match | Percentile:  25.0 | WEAKNESS


In [21]:
attack_process = ["Shots", "ShotsOnTarget", "Assists", "Crosses"]
defence_process = ["TacklesWon", "Interceptions", "FoulsCommitted"]

def diagnose(name):
    row = team[team["Squad"] == name].iloc[0]
    attack = row["Goals_permatch_percentile"]
    defence = row["GoalsConceded_permatch_percentile"]

    print(f"{name}: attack {attack:.0f} | defence {defence:.0f}")

    if attack >= defence:
        weaker, gap, process = "defence", attack - defence, defence_process
    else:
        weaker, gap, process = "attack", defence - attack, attack_process

    print(f"Weaker area: {weaker} (gap of {gap:.0f} percentile points)")
    print(f"Related stats for {weaker}:")
    for s in process:
        pct = row[s + "_permatch_percentile"]
        print(f"  {s:16} percentile {pct:5.1f}")

diagnose("Manchester Utd")

Manchester Utd: attack 83 | defence 64
Weaker area: defence (gap of 19 percentile points)
Related stats for defence:
  TacklesWon       percentile  83.9
  Interceptions    percentile  48.4
  FoulsCommitted   percentile  79.7


In [22]:
keeper = df[df["Pos"] == "GK"].groupby(["Squad", "Comp"]).agg(
    ShotsOnTargetFaced=("SoTA", "sum"),
    Saves=("Saves", "sum"),
).reset_index()

team = team.merge(keeper, on=["Squad", "Comp"], how="left")

team["ShotsOnTargetFaced_permatch"] = team["ShotsOnTargetFaced"] / team["Matches"]
team["SavePercent"] = team["Saves"] / team["ShotsOnTargetFaced"] * 100

team["ShotsOnTargetFaced_permatch_percentile"] = 100 - team["ShotsOnTargetFaced_permatch"].rank(pct=True) * 100
team["SavePercent_percentile"] = team["SavePercent"].rank(pct=True) * 100

team[team["Squad"] == "Manchester Utd"][["ShotsOnTargetFaced_permatch", "ShotsOnTargetFaced_permatch_percentile", "SavePercent", "SavePercent_percentile"]]

,ShotsOnTargetFaced_permatch,ShotsOnTargetFaced_permatch_percentile,SavePercent,SavePercent_percentile
56,3.631579,81.770833,65.942029,19.791667


In [23]:
def explain_defence(name):
    row = team[team["Squad"] == name].iloc[0]
    conceded = row["GoalsConceded_permatch_percentile"]
    faced = row["ShotsOnTargetFaced_permatch_percentile"]
    saves = row["SavePercent_percentile"]

    if conceded >= 75:
        print(f"{name}: strong defence, concedes fewer goals than most teams.")
    elif faced >= 50 and saves < 40:
        print(f"{name}: limits shots on target, but concedes more than the best defences. The gap is in goal, where saves are below most teams.")
    elif faced < 50 and saves >= 50:
        print(f"{name}: opponents get plenty of shots on target, so the problem is in front of the keeper, not in goal.")
    elif faced < 50 and saves < 50:
        print(f"{name}: allows many shots on target and saves fewer than most teams, so the problem is both.")
    else:
        print(f"{name}: no clear defensive problem in this data.")

explain_defence("Manchester Utd")

Manchester Utd: limits shots on target, but concedes more than the best defences. The gap is in goal, where saves are below most teams.


In [24]:
team_profile("Arsenal")

Arsenal (eng Premier League)

GoalsConceded      0.71 per match | Percentile:  99.0 | STRENGTH
Assists            1.34 per match | Percentile:  87.5 | STRENGTH
Goals              1.76 per match | Percentile:  84.4 | STRENGTH
Shots             14.53 per match | Percentile:  84.4 | STRENGTH
ShotsOnTarget      4.89 per match | Percentile:  82.3 | STRENGTH
FoulsCommitted    10.29 per match | Percentile:  78.1 | STRENGTH
Crosses           17.87 per match | Percentile:  58.9 | average
Offsides           1.61 per match | Percentile:  57.3 | average
TacklesWon         8.74 per match | Percentile:  19.8 | WEAKNESS
Interceptions      6.82 per match | Percentile:   9.4 | WEAKNESS


In [25]:
explain_defence("Arsenal")

Arsenal: strong defence, concedes fewer goals than most teams.


In [26]:
explain_defence("Manchester Utd")

worst = team.sort_values("GoalsConceded_permatch", ascending=False).head(5)["Squad"].tolist()
for t in worst:
    explain_defence(t)

Manchester Utd: limits shots on target, but concedes more than the best defences. The gap is in goal, where saves are below most teams.
Metz: allows many shots on target and saves fewer than most teams, so the problem is both.
Heidenheim: allows many shots on target and saves fewer than most teams, so the problem is both.
Wolfsburg: allows many shots on target and saves fewer than most teams, so the problem is both.
Burnley: allows many shots on target and saves fewer than most teams, so the problem is both.
Eintracht Frankfurt: allows many shots on target and saves fewer than most teams, so the problem is both.


In [29]:
def defence_label(row):
    conceded = row["GoalsConceded_permatch_percentile"]
    faced = row["ShotsOnTargetFaced_permatch_percentile"]
    saves = row["SavePercent_percentile"]
    if conceded >= 75:
        return "strong defence"
    elif faced >= 50 and saves < 40:
        return "gap in goal"
    elif faced < 50 and saves >= 50:
        return "problem in front of keeper"
    elif faced < 50 and saves < 50:
        return "both"
    else:
        return "average defence"

team["DefenceLabel"] = team.apply(defence_label, axis=1)

print(team["DefenceLabel"].value_counts())
team[team["DefenceLabel"] == "gap in goal"][["Squad", "Comp"]]

DefenceLabel
both                          25
strong defence                24
problem in front of keeper    23
gap in goal                   16
average defence                8
Name: count, dtype: int64


,Squad,Comp
0,Alavés,es La Liga
11,Bologna,it Serie A
30,Fulham,eng Premier League
37,Hellas Verona,it Serie A
45,Leeds United,eng Premier League
50,Liverpool,eng Premier League
51,Lorient,fr Ligue 1
56,Manchester Utd,eng Premier League
68,Paris FC,fr Ligue 1
72,RB Leipzig,de Bundesliga


In [28]:
print("Teams missing keeper data:", team["SavePercent"].isna().sum())

team[team["DefenceLabel"] == "no clear problem"][
    ["Squad", "Comp", "GoalsConceded_permatch_percentile",
     "ShotsOnTargetFaced_permatch_percentile", "SavePercent_percentile"]
]

Teams missing keeper data: 0


,Squad,Comp,GoalsConceded_permatch_percentile,ShotsOnTargetFaced_permatch_percentile,SavePercent_percentile
13,Brentford,eng Premier League,54.687500,61.458333,58.333333
18,Celta Vigo,es La Liga,72.916667,59.375000,80.208333
22,Crystal Palace,eng Premier League,58.333333,53.125000,52.083333
27,Everton,eng Premier League,64.062500,72.395833,41.666667
43,Le Havre,fr Ligue 1,68.229167,63.541667,73.958333
44,Lecce,it Serie A,64.062500,56.250000,61.458333
57,Marseille,fr Ligue 1,60.416667,73.958333,44.791667
65,Nottingham Forest,eng Premier League,58.333333,56.250000,57.291667


In [30]:
for short, name in {"Sh": "Shots", "SoT": "ShotsOnTarget"}.items():
    players[name + "_per90"] = players[short] / players["90s"]
    players[name + "_per90_percentile"] = players.groupby("PosGroup")[name + "_per90"].rank(pct=True) * 100

In [31]:
score_stats = {
    "FW": ["Goals", "Assists", "Shots", "ShotsOnTarget"],
    "MF": ["Goals", "Assists", "TacklesWon", "Interceptions", "FoulsDrawn"],
    "DF": ["TacklesWon", "Interceptions", "FoulsCommitted", "Assists"],
}

def player_score(row):
    stats = score_stats.get(row["PosGroup"])
    if stats is None:
        return None
    return sum(row[s + "_per90_percentile"] for s in stats) / len(stats)

players["Score"] = players.apply(player_score, axis=1)

fw = players[players["PosGroup"] == "FW"]
fw[["Player", "Squad", "Score"]].sort_values("Score", ascending=False).head(10)

,Player,Squad,Score
2656,Deniz Undav,Stuttgart,96.753247
667,Ousmane Dembélé,Paris Saint-Germain,95.454545
1104,Erling Haaland,Manchester City,94.642857
1319,Harry Kane,Bayern Munich,94.588745
1701,Kylian Mbappé,Real Madrid,93.614719
1663,Lautaro Martínez,Inter,92.857143
1421,Nikola Krstović,Atalanta,90.476190
2588,Marcus Thuram,Inter,89.177489
1432,Khvicha Kvaratskhelia,Paris Saint-Germain,88.906926
1610,Donyell Malen,Roma,88.203463


In [32]:
def underperformers(team_name, n=5):
    t = players[(players["Squad"] == team_name) & (players["Score"].notna())]
    return t[["Player", "PosGroup", "Min", "Score"]].sort_values("Score").head(n)

underperformers("Manchester Utd")

,Player,PosGroup,Min,Score
1705,Bryan Mbeumo,MF,2609,38.892045
1664,Lisandro Martínez,DF,1233,40.524575
1695,Noussair Mazraoui,DF,985,46.526465
691,Amad Diallo,MF,2341,47.102273
1498,Matthijs de Ligt,DF,1170,48.818526


In [33]:
underperformers("Chelsea")

,Player,PosGroup,Min,Score
661,Liam Delap,FW,1107,32.792208
1898,Pedro Neto,MF,2631,37.684659
901,Wesley Fofana,DF,1722,46.077505
517,Trevoh Chalobah,DF,2783,47.424386
2048,Cole Palmer,MF,1955,48.835227


In [34]:
def role(pos):
    if pos == "MF,FW":
        return "AM"
    return pos.split(",")[0]

players["Role"] = players["Pos"].apply(role)

all_stats = ["Goals", "Assists", "Shots", "ShotsOnTarget", "Interceptions",
             "TacklesWon", "Crosses", "FoulsCommitted", "FoulsDrawn", "Offsides"]
lower_is_better = ["FoulsCommitted", "Offsides"]

for s in all_stats:
    col = s + "_per90"
    pct = players.groupby("Role")[col].rank(pct=True) * 100
    players[col + "_percentile"] = 100 - pct if s in lower_is_better else pct

score_stats = {
    "FW": ["Goals", "Assists", "Shots", "ShotsOnTarget"],
    "AM": ["Goals", "Assists", "Shots", "ShotsOnTarget", "FoulsDrawn"],
    "MF": ["Goals", "Assists", "TacklesWon", "Interceptions", "FoulsDrawn"],
    "DF": ["TacklesWon", "Interceptions", "FoulsCommitted", "Assists"],
}

def player_score(row):
    stats = score_stats.get(row["Role"])
    if stats is None:
        return None
    return sum(row[s + "_per90_percentile"] for s in stats) / len(stats)

players["Score"] = players.apply(player_score, axis=1)

def underperformers(team_name, below=40):
    t = players[(players["Squad"] == team_name) & (players["Score"] < below)]
    return t[["Player", "Role", "Min", "Score"]].sort_values("Score")

print(players["Role"].value_counts())
underperformers("Chelsea")

Role
MF    555
DF    529
FW    231
AM    149
GK    114
Name: count, dtype: int64


,Player,Role,Min,Score
661,Liam Delap,FW,1107,32.792208
1898,Pedro Neto,AM,2631,35.637584


In [35]:
role_names = {"FW": "Forward", "MF": "Midfielder", "DF": "Defender", "GK": "Goalkeeper"}

def role(pos):
    if pos == "MF,FW":
        return "Attacking Midfielder"
    return role_names.get(pos.split(",")[0], pos)

players["Role"] = players["Pos"].apply(role)

all_stats = ["Goals", "Assists", "Shots", "ShotsOnTarget", "Interceptions",
             "TacklesWon", "Crosses", "FoulsCommitted", "FoulsDrawn", "Offsides"]
lower_is_better = ["FoulsCommitted", "Offsides"]

for s in all_stats:
    col = s + "_per90"
    pct = players.groupby("Role")[col].rank(pct=True) * 100
    players[col + "_percentile"] = 100 - pct if s in lower_is_better else pct

score_stats = {
    "Forward": ["Goals", "Assists", "Shots", "ShotsOnTarget"],
    "Attacking Midfielder": ["Goals", "Assists", "Shots", "ShotsOnTarget", "FoulsDrawn"],
    "Midfielder": ["Goals", "Assists", "TacklesWon", "Interceptions", "FoulsDrawn"],
    "Defender": ["TacklesWon", "Interceptions", "FoulsCommitted", "Assists"],
}

def player_score(row):
    stats = score_stats.get(row["Role"])
    if stats is None:
        return None
    return sum(row[s + "_per90_percentile"] for s in stats) / len(stats)

players["Score"] = players.apply(player_score, axis=1)

def underperformers(team_name, below=40):
    t = players[(players["Squad"] == team_name) & (players["Score"] < below)]
    return t[["Player", "Role", "Min", "Score"]].sort_values("Score")

print(players["Role"].value_counts())
underperformers("Chelsea")

Role
Midfielder              555
Defender                529
Forward                 231
Attacking Midfielder    149
Goalkeeper              114
Name: count, dtype: int64


,Player,Role,Min,Score
661,Liam Delap,Forward,1107,32.792208
1898,Pedro Neto,Attacking Midfielder,2631,35.637584


In [36]:
gk = players[players["Role"] == "Goalkeeper"].copy()

gk["SavePercent_pct"] = gk["Save%"].rank(pct=True) * 100
gk["GoalsAgainst_pct"] = 100 - gk["GA90"].rank(pct=True) * 100
gk["CleanSheet_pct"] = gk["CS%"].rank(pct=True) * 100

gk["Score"] = gk[["SavePercent_pct", "GoalsAgainst_pct", "CleanSheet_pct"]].mean(axis=1)
players.loc[gk.index, "Score"] = gk["Score"]

gk[["Player", "Squad", "Min", "Save%", "GA90", "CS%", "Score"]].sort_values("Score", ascending=False).head(10)

,Player,Squad,Min,Save%,GA90,CS%,Score
960,Joan García,Barcelona,2700,80.4,0.70,50.0,97.953216
413,Jean Butez,Como,3420,78.2,0.76,50.0,96.491228
2533,Mile Svilar,Roma,3420,78.1,0.82,47.4,95.029240
2152,Ivan Provedel,Lazio,2430,78.6,1.00,44.4,91.812865
2309,Matvei Safonov,Paris Saint-Germain,1350,73.6,0.93,53.3,90.350877
600,Thibaut Courtois,Real Madrid,2880,74.5,0.87,40.6,89.181287
1370,Gregor Kobel,Dortmund,3060,75.2,1.00,44.1,89.035088
532,Lucas Chevalier,Paris Saint-Germain,1530,72.1,0.76,52.9,88.596491
1591,Mike Maignan,Milan,3295,77.6,0.96,37.8,88.011696
479,Marco Carnesecchi,Atalanta,3330,77.1,0.95,35.1,87.134503


In [37]:
def recommend(team_name, role, n=5):
    squad = players[(players["Squad"] == team_name) & (players["Role"] == role)]
    if squad.empty:
        print(f"No {role} with enough minutes found for {team_name}.")
        return
    current_best = squad["Score"].max()
    print(f"{team_name}'s best {role} scores {current_best:.1f}")
    pool = players[(players["Role"] == role) & (players["Squad"] != team_name) & (players["Score"] > current_best)]
    return pool[["Player", "Squad", "Age", "Min", "Score"]].sort_values("Score", ascending=False).head(n)

recommend("Manchester Utd", "Goalkeeper")

Manchester Utd's best Goalkeeper scores 55.7


,Player,Squad,Age,Min,Score
960,Joan García,Barcelona,24.0,2700,97.953216
413,Jean Butez,Como,30.0,3420,96.491228
2533,Mile Svilar,Roma,25.0,3420,95.029240
2152,Ivan Provedel,Lazio,31.0,2430,91.812865
2309,Matvei Safonov,Paris Saint-Germain,26.0,1350,90.350877
